# Clase 09 — Despliegue y publicación

**Curso:** MLOps y Monitorización con Python · **Bloque B**
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/09_despliegue_publicacion.md`

## Qué verás en esta clase

- Los tres **entornos** de un sistema real (desarrollo, staging, producción) y para qué sirve cada uno, con un ejemplo concreto.
- Reentrenar las dos versiones del modelo de churn (v1 sin balancear, v2 balanceada) de las clases 4, 7 y 8.
- Construir, sobre la API de la clase 7, dos versiones activas al mismo tiempo (`/predecir_v1`, `/predecir_v2`).
- Simular en código real tres estrategias de despliegue con un generador de tráfico sintético reproducible: **canary**, **shadow** y **A/B**.
- Por qué el A/B usa un **hash determinista** del `id_cliente` y no un sorteo aleatorio puro.
- Implementar y probar una función de **rollback** con un criterio de decisión explícito y un log real de la acción.

Todo ocurre dentro de una carpeta de trabajo efímera (`proyecto_churn/`), creada al principio y **eliminada al final** de este mismo notebook. No se usa MLflow en esta clase: no hace falta para simular tráfico y decidir un rollback.

## Prerequisitos

- De la clase 7: la API FastAPI (`Cliente`, `TestClient`) que este notebook amplía a dos versiones.
- De la clase 8: el registro de modelos con alias `candidato`/`producción`/`archivado` -- esta clase simula, con tráfico real, la decisión que justificaría promover uno u otro.
- De las clases 4 y 5: la configuración ganadora del modelo (regresión logística, `class_weight="balanced"`, `C=1,0`) y su contraste con la versión sin balancear.
- Python básico: funciones, diccionarios, `Counter`; no hace falta haber usado `hashlib` ni pruebas estadísticas antes.

> **Cómo usar este notebook:** ejecuta las celdas en orden, de arriba a abajo, una sola vez. Tarda menos de un minuto en total.

---
## Instalación

Instala solo lo que falte (FastAPI, httpx, uvicorn, scipy).

In [ ]:
import warnings

# Silenciamos un aviso de compatibilidad de starlette/httpx (fijo, no aporta nada aquí) para que
# la salida de este notebook no dependa de detalles internos de versiones de librerías
warnings.filterwarnings("ignore", category=DeprecationWarning)

import subprocess
import sys


def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("scipy", "scipy")
asegurar("joblib", "joblib")
asegurar("fastapi", "fastapi")
asegurar("httpx", "httpx")
asegurar("uvicorn", "uvicorn")
print("Librerías listas.")

---
## Celda 0 — Setup

In [ ]:
import os
import shutil
import hashlib
import json
from collections import Counter

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]
COLUMNAS_ENTRADA = COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS

print("Librerías y funciones auxiliares listas.")

---
## Carpeta de trabajo efímera

In [ ]:
CARPETA = "proyecto_churn"

# Limpiamos y recreamos la carpeta para que el notebook sea repetible desde cero (misma convención de las clases 7 y 8)
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(CARPETA)
print("Carpeta de trabajo creada:", os.path.abspath(CARPETA))

---
## 1. Los tres entornos: desarrollo, staging, producción

Antes de repartir tráfico entre versiones, conviene distinguir DÓNDE corre cada cosa. Los tres entornos comparten el mismo esquema de datos y, en un proyecto serio, el mismo código -- lo que cambia es el volumen de datos, quién los usa y el riesgo de un fallo:

| Entorno | Para qué sirve | Quién lo usa | Riesgo de un fallo |
|---|---|---|---|
| **Desarrollo** | Iterar rápido: probar una idea, depurar, ejecutar el notebook de esta clase | quien programa | bajo -- solo afecta a quien lo ejecuta |
| **Staging** | Una réplica de producción con datos realistas, para probar el `Dockerfile`/`ci.yml` de la clase 8 antes de tocar tráfico real | el equipo, antes de publicar | medio -- si falla, no llega a un cliente |
| **Producción** | Sirve peticiones reales de clientes reales | los clientes, sin saberlo | alto -- un fallo aquí es un incidente de verdad |

El notebook lo hace concreto: `desarrollo` usa una muestra de 50 filas de `churn.csv`, `staging` usa el 20 % de las filas, y `producción` usa el conjunto completo -- MISMAS columnas en los tres, para que un `Cliente` validado en desarrollo lo siga siendo en producción.

In [ ]:
RUTA_CSV = cargar_datos("churn.csv")
df = pd.read_csv(RUTA_CSV)

# Un mismo esquema (las mismas columnas), tres cantidades de datos distintas -- una diferencia real y verificable entre entornos
ENTORNOS = {
    "desarrollo": {
        "datos": df.sample(n=50, random_state=SEMILLA),
        "descripcion": "portátil del desarrollador, subconjunto pequeño, iteración rápida",
    },
    "staging": {
        "datos": df.sample(frac=0.2, random_state=SEMILLA),
        "descripcion": "réplica de producción con una muestra realista de datos, sin tráfico de clientes reales",
    },
    "produccion": {
        "datos": df,
        "descripcion": "sirve peticiones reales; cualquier fallo aquí afecta a un cliente de verdad",
    },
}

for nombre, info in ENTORNOS.items():
    print(f"entorno {nombre}: {len(info['datos'])} filas -- {info['descripcion']}")

assert set(ENTORNOS["staging"]["datos"].columns) == set(ENTORNOS["produccion"]["datos"].columns), \
    "staging debe compartir el mismo esquema que producción"
assert len(ENTORNOS["desarrollo"]["datos"]) < len(ENTORNOS["staging"]["datos"]) < len(ENTORNOS["produccion"]["datos"])
print("Los tres entornos comparten esquema; solo cambian el volumen de datos y el riesgo de un fallo.")

> **Regla práctica:** si un cambio no se ha probado en staging con datos que se parezcan a los reales, no está listo para producción -- por rápido que haya ido en desarrollo. Staging existe precisamente para atrapar el "en mi portátil funcionaba" antes de que le llegue a un cliente.

---
## 2. Reentrenar v1 y v2 (como en las clases 4, 7 y 8)

Igual que en las clases anteriores, este notebook reentrena de cero, dentro de su propia carpeta efímera: no depende de ningún artefacto guardado antes. `v1` es la configuración sin balancear (la que, en la clase 8, era la versión "ya en producción" antes de la promoción); `v2` es la configuración ganadora balanceada -- el candidato que esta clase despliega con cuidado, en vez de promover de golpe como hizo directamente la clase 8 al reasignar el alias.

In [ ]:
X = df[COLUMNAS_ENTRADA]
y = df["abandono"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA, stratify=y
)

preprocesador = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUMNAS_CATEGORICAS),
    ("numericas", StandardScaler(), COLUMNAS_NUMERICAS),
])

# v1: la configuración SIN balancear -- la que, en la clase 8, "ya estaba en producción" antes de la promoción
clasificador_v1 = LogisticRegression(random_state=SEMILLA, max_iter=1000)
pipeline_v1 = Pipeline([("preprocesado", preprocesador), ("clasificador", clasificador_v1)])
pipeline_v1.fit(X_train, y_train)
f1_v1 = round(f1_score(y_test, pipeline_v1.predict(X_test)), 4)
auc_v1 = round(roc_auc_score(y_test, pipeline_v1.predict_proba(X_test)[:, 1]), 4)
acc_v1 = round(accuracy_score(y_test, pipeline_v1.predict(X_test)), 4)

# v2: la configuración ganadora (balanceada) de las clases 4, 5, 7 y 8 -- el "candidato" que esta clase despliega con cuidado
clasificador_v2 = LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000, C=1.0)
pipeline_v2 = Pipeline([("preprocesado", preprocesador), ("clasificador", clasificador_v2)])
pipeline_v2.fit(X_train, y_train)
f1_v2 = round(f1_score(y_test, pipeline_v2.predict(X_test)), 4)
auc_v2 = round(roc_auc_score(y_test, pipeline_v2.predict_proba(X_test)[:, 1]), 4)
acc_v2 = round(accuracy_score(y_test, pipeline_v2.predict(X_test)), 4)

MODELOS = {"v1": pipeline_v1, "v2": pipeline_v2}

print("v1 (sin balancear, en producción hasta hoy) -- F1:", f1_v1, "AUC:", auc_v1, "exactitud:", acc_v1)
print("v2 (balanceada, candidata)        -- F1:", f1_v2, "AUC:", auc_v2, "exactitud:", acc_v2)
assert f1_v2 > f1_v1 and auc_v1 > 0.6, "el reentrenamiento no reprodujo los modelos esperados"

---
## 3. Construir la API con dos versiones activas

Sobre la misma `Cliente` y el mismo patrón de la clase 7, este notebook expone DOS endpoints -- `/predecir_v1` y `/predecir_v2` -- que comparten el mismo proceso y el mismo `TestClient`, simulando dos versiones desplegadas al mismo tiempo detrás de un enrutador de tráfico.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from fastapi import FastAPI
from fastapi.testclient import TestClient


class Cliente(BaseModel):
    """El mismo modelo Pydantic de la clase 7: exactamente las columnas de entrada de churn.csv."""
    id_cliente: str
    antiguedad_meses: int = Field(ge=0)
    plan: Literal["basico", "estandar", "premium"]
    contrato: Literal["mensual", "anual"]
    uso_mensual_horas: float = Field(ge=0)
    tickets_soporte: int = Field(ge=0)
    metodo_pago: Literal["domiciliacion", "tarjeta", "transferencia"]
    factura_mensual: float = Field(ge=0)


def cliente_a_fila(cliente):
    return pd.DataFrame([cliente.model_dump()])[COLUMNAS_ENTRADA]


def respuesta_de(nombre_version, cliente):
    fila = cliente_a_fila(cliente)
    probabilidad = float(MODELOS[nombre_version].predict_proba(fila)[0, 1])
    return {
        "id_cliente": cliente.id_cliente,
        "version": nombre_version,
        "probabilidad_abandono": round(probabilidad, 4),
        "abandono_previsto": int(probabilidad >= 0.5),
    }


app = FastAPI(title="API de churn (clase 7), con dos versiones activas")


@app.post("/predecir_v1")
def predecir_v1(cliente: Cliente):
    return respuesta_de("v1", cliente)


@app.post("/predecir_v2")
def predecir_v2(cliente: Cliente):
    return respuesta_de("v2", cliente)


# TestClient, como en la clase 7: sin puertos ni procesos reales, determinista
cliente_api = TestClient(app)

cliente_prueba = X_test.iloc[0].to_dict()
cliente_prueba["id_cliente"] = "C_TEST_0001"
r1 = cliente_api.post("/predecir_v1", json=cliente_prueba)
r2 = cliente_api.post("/predecir_v2", json=cliente_prueba)
print("/predecir_v1 ->", r1.status_code, r1.json())
print("/predecir_v2 ->", r2.status_code, r2.json())
assert r1.status_code == 200 and r2.status_code == 200
assert r1.json()["version"] == "v1" and r2.json()["version"] == "v2"
print("Las dos versiones de la API responden, en el mismo proceso, listas para repartir tráfico entre ellas.")

---
## 4. Un generador de tráfico sintético reproducible

Para comparar estrategias necesitamos peticiones -- muchas, y siempre las MISMAS entre una ejecución y otra. El notebook muestrea, sin reemplazo y con una semilla fija (`SEMILLA_TRAFICO=7`, distinta de la semilla del modelo), 400 filas de `churn.csv` y las convierte en peticiones válidas para `Cliente` (sin la columna `abandono`, que es la etiqueta a predecir, no un dato de entrada).

In [ ]:
N_TRAFICO = 400
# Semilla fija y distinta de SEMILLA (la del modelo): así el generador de tráfico es reproducible por sí mismo
SEMILLA_TRAFICO = 7
rng_trafico = np.random.default_rng(SEMILLA_TRAFICO)
indices_trafico = rng_trafico.choice(df.index.values, size=N_TRAFICO, replace=False)
trafico_df = df.loc[indices_trafico].reset_index(drop=True)
peticiones = trafico_df[["id_cliente"] + COLUMNAS_ENTRADA].to_dict(orient="records")

print("Tráfico sintético generado:", len(peticiones), "peticiones, muestreadas sin reemplazo de churn.csv con semilla fija.")
print("primera petición:", peticiones[0])

---
## 5. Estrategia 1 — Canary: una fracción pequeña a la versión nueva

En un despliegue **canary**, la versión candidata recibe solo una fracción pequeña del tráfico (aquí, el 8 %); el resto sigue en la versión estable. Si algo va mal, el daño queda limitado a esa fracción. La decisión de a qué versión va cada petición se toma con un generador aleatorio (`numpy.random.default_rng`) con semilla fija: aleatorio en el sentido de "no depende del contenido de la petición", pero reproducible entre ejecuciones.

In [ ]:
P_CANARY = 0.08  # 8 % del tráfico a la versión candidata
SEMILLA_CANARY = 123
rng_canary = np.random.default_rng(SEMILLA_CANARY)


def enrutar_canario(rng):
    """Sortea, con la MISMA semilla en cada ejecución, si esta petición va a v2 (candidata) o a v1 (estable)."""
    return "v2" if rng.random() < P_CANARY else "v1"


conteo_canary = Counter()
for peticion in peticiones:
    version = enrutar_canario(rng_canary)
    conteo_canary[version] += 1
    respuesta = cliente_api.post(f"/predecir_{version}", json=peticion)
    assert respuesta.status_code == 200

pct_v2_canary = round(100 * conteo_canary["v2"] / N_TRAFICO, 2)
print("Reparto canary sobre", N_TRAFICO, "peticiones:", dict(conteo_canary))
print(f"-> {pct_v2_canary} % fue a la versión candidata (v2), el resto a la versión estable (v1) -- objetivo: {P_CANARY:.0%}.")
assert conteo_canary["v1"] + conteo_canary["v2"] == N_TRAFICO

**¿Qué observas?** El reparto real (recalculado en el notebook) queda cerca del 8 % objetivo, pero no exactamente en él -- 400 sorteos independientes con probabilidad 0,08 no dan EXACTAMENTE 32 casos, por la misma razón por la que lanzar una moneda 400 veces no da exactamente 200 caras. Con más tráfico, el reparto observado se acerca más al objetivo.

---
## 6. Estrategia 2 — Shadow: la misma petición, dos respuestas, una sola cuenta

En un despliegue **shadow**, la versión candidata (v2) recibe una COPIA de cada petición, pero su respuesta nunca llega al usuario: solo se compara con la de la versión en uso (v1). Es la estrategia de menor riesgo posible -- v2 nunca decide nada por sí sola -- a costa de duplicar el trabajo de cómputo (cada petición se procesa dos veces).

In [ ]:
# En shadow, TODAS las peticiones se envían también a v2, pero la respuesta que "ve el usuario" siempre es la de v1
acuerdos = 0
desacuerdos_muestra = []
for peticion in peticiones:
    respuesta_usuario = cliente_api.post("/predecir_v1", json=peticion).json()  # esto es lo único que llega al usuario
    respuesta_sombra = cliente_api.post("/predecir_v2", json=peticion).json()   # esto solo se registra para comparar, nunca se devuelve
    if respuesta_usuario["abandono_previsto"] == respuesta_sombra["abandono_previsto"]:
        acuerdos += 1
    elif len(desacuerdos_muestra) < 3:
        desacuerdos_muestra.append((peticion["id_cliente"], respuesta_usuario["abandono_previsto"], respuesta_sombra["abandono_previsto"]))

tasa_acuerdo = round(acuerdos / N_TRAFICO, 4)
print("Acuerdos v1/v2 en shadow:", acuerdos, "de", N_TRAFICO, f"-> tasa de acuerdo: {tasa_acuerdo:.2%}")
print("ejemplos de desacuerdo (id_cliente, previsto v1, previsto v2):", desacuerdos_muestra)
print("El usuario NUNCA vio la respuesta de v2 en esta sección: shadow mide sin ningún riesgo para nadie.")
assert 0.0 <= tasa_acuerdo <= 1.0

**¿Qué observas?** La tasa de acuerdo recalculada es notablemente menor que 1: v1 (sin balancear) casi nunca predice abandono (su F1 en la sección 2 es prácticamente nulo), mientras que v2 predice abandono con más frecuencia. Ese desacuerdo NO es un error del notebook: es la diferencia real de comportamiento entre las dos versiones, exactamente el tipo de señal que shadow está pensado para exponer antes de decidir una promoción.

---
## 7. Estrategia 3 — A/B: reparto determinista por `id_cliente`

En un despliegue **A/B**, las dos versiones están activas y sirven tráfico real al mismo tiempo (a diferencia de canary, donde una es claramente la "minoritaria"). La pregunta que responde no es "¿falla v2?" sino "¿v2 es MEJOR para el negocio?", medido con una métrica de negocio -- aquí, una tasa de conversión ficticia (por ejemplo, aceptar una oferta de retención), no una métrica del modelo como F1.

El reparto entre grupos usa un **hash determinista** del `id_cliente` (MD5, en este notebook), NO un sorteo aleatorio puro en cada petición. La razón es de producto, no técnica: si el mismo cliente pudiera caer en un grupo distinto en cada visita, vería precios, textos o decisiones distintas de una sesión a otra -- una experiencia inconsistente que generaría quejas y contaminaría la propia medición (¿mejoró la conversión por la versión, o porque el mismo cliente vio ambas y se confundió?). El hash resuelve esto sin necesitar guardar en ninguna base de datos "a qué grupo pertenece cada cliente": la misma entrada produce siempre la misma salida.

In [ ]:
def grupo_ab(id_cliente):
    """Hash MD5 del id_cliente, no un sorteo aleatorio: el MISMO id_cliente cae SIEMPRE en el mismo grupo."""
    digest = int(hashlib.md5(id_cliente.encode()).hexdigest(), 16)
    return "v1_control" if digest % 2 == 0 else "v2_tratamiento"


# Demostración: el hash es determinista (se repite en dos llamadas); un sorteo aleatorio con random.random() no lo sería
un_id = trafico_df.loc[0, "id_cliente"]
assert grupo_ab(un_id) == grupo_ab(un_id), "el hash debería dar siempre el mismo grupo para el mismo id_cliente"
print(f"'{un_id}' cae en el grupo", grupo_ab(un_id), "en dos llamadas seguidas -- siempre el mismo.")

### 7.1 Repartir toda la base de clientes y simular una métrica de negocio

In [ ]:
# Asignamos el grupo a TODA la base de clientes (no solo al tráfico de la sección 4): así, cada cliente que vuelva
# mañana sigue viendo la misma versión que hoy -- la razón de negocio detrás de usar un hash y no un número aleatorio
grupos = df["id_cliente"].astype(str).map(grupo_ab)
n_control = int((grupos == "v1_control").sum())
n_tratamiento = int((grupos == "v2_tratamiento").sum())
print("tamaño del grupo v1_control:", n_control, "| tamaño del grupo v2_tratamiento:", n_tratamiento)
assert n_control + n_tratamiento == len(df)

# Métrica de negocio simulada: tasa de conversión (por ejemplo, aceptar una oferta de retención), NO una métrica del modelo
# con probabilidades distintas por grupo (datos ficticios, creados para este ejercicio) y semilla fija
SEMILLA_CONVERSION = 42
rng_conversion = np.random.default_rng(SEMILLA_CONVERSION)
PROB_CONVERSION_CONTROL = 0.20
PROB_CONVERSION_TRATAMIENTO = 0.245
conversiones_control = rng_conversion.binomial(1, PROB_CONVERSION_CONTROL, n_control)
conversiones_tratamiento = rng_conversion.binomial(1, PROB_CONVERSION_TRATAMIENTO, n_tratamiento)

tasa_control = conversiones_control.mean()
tasa_tratamiento = conversiones_tratamiento.mean()
print(f"tasa de conversión v1_control: {tasa_control:.4f} ({int(conversiones_control.sum())}/{n_control})")
print(f"tasa de conversión v2_tratamiento: {tasa_tratamiento:.4f} ({int(conversiones_tratamiento.sum())}/{n_tratamiento})")

### 7.2 Comparar los dos grupos con un test estadístico real

In [ ]:
# Tabla de contingencia 2x2 (convierte/no convierte x control/tratamiento) y chi-cuadrado de independencia
tabla_contingencia = [
    [int(conversiones_control.sum()), n_control - int(conversiones_control.sum())],
    [int(conversiones_tratamiento.sum()), n_tratamiento - int(conversiones_tratamiento.sum())],
]
chi2, valor_p, grados_libertad, _ = chi2_contingency(tabla_contingencia, correction=False)
print("tabla de contingencia [convierte, no convierte] x [control, tratamiento]:", tabla_contingencia)
print("chi-cuadrado:", round(chi2, 4), "valor p:", round(valor_p, 4))

# Además, la diferencia de proporciones con un intervalo de confianza del 95 % (aproximación normal de Wald)
diferencia = tasa_tratamiento - tasa_control
error_estandar_diferencia = np.sqrt(
    tasa_control * (1 - tasa_control) / n_control + tasa_tratamiento * (1 - tasa_tratamiento) / n_tratamiento
)
ic_bajo = diferencia - 1.96 * error_estandar_diferencia
ic_alto = diferencia + 1.96 * error_estandar_diferencia
print(f"diferencia (tratamiento - control): {diferencia:.4f}, IC 95 %: [{ic_bajo:.4f}, {ic_alto:.4f}]")

ALFA = 0.05
es_significativo = valor_p < ALFA
print(f"¿diferencia estadísticamente significativa (p < {ALFA})?:", es_significativo)
assert (ic_bajo > 0) == es_significativo, "el IC y el valor p deberían coincidir en su conclusión"
print("El intervalo de confianza NO incluye 0 y el valor p es menor que 0,05: en este ejemplo ilustrativo, el grupo con v2 convierte más, y no parece azar.")

> **Atención:** en este ejemplo ilustrativo, las tasas de conversión son datos SIMULADOS (con `rng_conversion.binomial`, semilla fija) -- no salen de `churn.csv` ni de ningún dato real del curso, porque el dataset no incluye ninguna métrica de conversión. Lo real y verificable aquí es el CÁLCULO del test estadístico sobre esos datos simulados, no la cifra de negocio en sí.

---
## 8. Comparar las tres estrategias

| | Canary | Shadow | A/B |
|---|---|---|---|
| ¿Quién ve la respuesta de v2? | una fracción pequeña de usuarios reales | nadie (solo un log de comparación) | la mitad (aprox.) de los usuarios reales |
| Riesgo si v2 tiene un fallo | limitado a esa fracción | ninguno | limitado al grupo tratamiento |
| Qué mide | ¿v2 falla o se comporta mal en producción real? | ¿en qué se diferencian las predicciones de v1 y v2? | ¿v2 es MEJOR para una métrica de negocio? |
| Coste de cómputo | el normal (cada petición va a una sola versión) | el doble (cada petición se procesa dos veces) | el normal |
| Reparto | aleatorio con semilla fija | ninguno -- todo el tráfico se duplica | determinista (hash de `id_cliente`) |

**Regla práctica:** shadow primero (riesgo cero, para saber si v2 se comporta razonablemente distinto), canary después (riesgo acotado, para confirmar que no falla con tráfico real), A/B al final (cuando ya se confía en que v2 no falla, y lo que queda por decidir es si es mejor para el negocio).

---
## 9. Rollback: volver atrás cuando una métrica empeora

Un **rollback** vuelve a la versión anterior cuando la nueva causa un problema. Para que no sea una decisión improvisada, hace falta un **criterio explícito**: una métrica concreta, una versión de referencia y un umbral. Aquí: si el F1 medido sobre un lote reciente cae más de un 15 % (relativo) respecto al F1 ya validado, se revierte a la versión anterior.

In [ ]:
UMBRAL_CAIDA_RELATIVA = 0.15  # 15 %: criterio de decisión explícito de este notebook


def decidir_rollback(metrica_base, metrica_actual, umbral_relativo=UMBRAL_CAIDA_RELATIVA):
    """Devuelve (debe_revertir, caida_relativa, motivo). "metrica_base" es la métrica ya validada (p.ej. F1 en el conjunto de prueba); "metrica_actual" es la misma métrica medida sobre tráfico más reciente."""
    caida_relativa = (metrica_base - metrica_actual) / metrica_base
    debe_revertir = caida_relativa > umbral_relativo
    if debe_revertir:
        motivo = f"caída relativa de {caida_relativa:.2%} SUPERA el umbral de {umbral_relativo:.0%}"
    else:
        motivo = f"caída relativa de {caida_relativa:.2%} NO supera el umbral de {umbral_relativo:.0%}"
    return debe_revertir, round(caida_relativa, 4), motivo


def registrar_decision(version_actual, version_anterior, debe_revertir, caida_relativa, motivo):
    """Un log real de la decisión (sin reloj de pared, para que sea reproducible): qué se decidió, sobre qué versión y por qué."""
    entrada = {
        "accion": "rollback" if debe_revertir else "mantener_version_actual",
        "version_en_uso_antes_de_decidir": version_actual,
        "version_restaurada": version_anterior if debe_revertir else version_actual,
        "caida_relativa": caida_relativa,
        "umbral": UMBRAL_CAIDA_RELATIVA,
        "motivo": motivo,
    }
    print(json.dumps(entrada, ensure_ascii=False))
    return entrada

### 9.1 Caso normal: una variación pequeña NO dispara el rollback

In [ ]:
# Caso 1: un lote de tráfico reciente con una variación pequeña y normal (15 % de las filas con un dato de uso ausente, por ejemplo un sensor que falla a veces)
rng_variacion = np.random.default_rng(SEMILLA)
indices_afectados_leve = rng_variacion.choice(X_test.index, size=int(len(X_test) * 0.15), replace=False)
X_test_variacion_leve = X_test.copy()
X_test_variacion_leve.loc[indices_afectados_leve, "uso_mensual_horas"] = 0.0
f1_v2_variacion_leve = round(f1_score(y_test, pipeline_v2.predict(X_test_variacion_leve)), 4)
print("F1 de v2 sobre el lote con variación leve:", f1_v2_variacion_leve, "(base:", f1_v2, ")")

debe_revertir_1, caida_1, motivo_1 = decidir_rollback(f1_v2, f1_v2_variacion_leve)
registrar_decision("v2", "v1", debe_revertir_1, caida_1, motivo_1)
assert debe_revertir_1 is False, "una variación tan pequeña no debería disparar un rollback"

### 9.2 Caso degradado: un fallo real de datos SÍ dispara el rollback

In [ ]:
# Caso 2: un fallo real de la canalización de features -- "uso_mensual_horas" llega siempre en 0 para TODO el lote (p.ej. el servicio que la calcula se cayó)
X_test_degradado = X_test.copy()
X_test_degradado["uso_mensual_horas"] = 0.0
f1_v2_degradado = round(f1_score(y_test, pipeline_v2.predict(X_test_degradado)), 4)
print("F1 de v2 sobre el lote degradado:", f1_v2_degradado, "(base:", f1_v2, ")")

debe_revertir_2, caida_2, motivo_2 = decidir_rollback(f1_v2, f1_v2_degradado)
entrada_log = registrar_decision("v2", "v1", debe_revertir_2, caida_2, motivo_2)
assert debe_revertir_2 is True, "esta caída debería disparar el rollback"
assert entrada_log["accion"] == "rollback" and entrada_log["version_restaurada"] == "v1"
print("Rollback ejecutado: el tráfico vuelve a v1 hasta que se corrija la canalización de features y se repita el despliegue gradual desde la sección 5.")

**¿Qué observas?** La MISMA función `decidir_rollback` con el MISMO umbral (15 %) da dos decisiones distintas porque la caída real es distinta: ~4 % en el caso normal (no dispara nada), ~21 % en el caso degradado (dispara el rollback). El criterio no cambia; lo que cambia es el dato que entra. El log (una línea JSON) queda como evidencia de qué se decidió y por qué, sin depender de ningún reloj de sistema.

---
## 10. Errores comunes

1. **Repartir el tráfico A/B con un sorteo aleatorio en cada petición, en vez de un hash del `id_cliente`**: el mismo cliente vería una versión distinta en cada visita, rompiendo la experiencia y contaminando la propia métrica de negocio (sección 7).
2. **Confundir shadow con canary**: en canary, v2 SÍ decide (para una fracción de usuarios); en shadow, v2 NUNCA decide -- solo se compara. Tratarlos como sinónimos lleva a subestimar el riesgo real de un despliegue canary.
3. **Promover un candidato a producción solo porque "no ha fallado" en canary**, sin mirar una métrica de negocio (A/B) ni comparar contra la versión anterior con un test estadístico: "no ha fallado" no es lo mismo que "es mejor".
4. **Fijar el umbral de rollback mirando los datos DESPUÉS de ver el problema** ("ajustemos el umbral para que hoy no dispare"): el umbral debe decidirse ANTES, con la métrica base ya validada (sección 9), o deja de ser un criterio y pasa a ser una excusa.
5. **Ignorar el tamaño de cada grupo en un test A/B**: con muestras muy pequeñas, hasta una diferencia real puede no ser estadísticamente significativa (intervalo de confianza ancho, que incluye el 0) -- la sección 7.2 comprueba esto explícitamente con el intervalo de confianza, no solo con el valor p.

---
## 11. Checklist antes de dar un despliegue por listo

- [ ] Sé para qué sirve cada entorno (desarrollo, staging, producción) y no confundo probar en staging con estar en producción.
- [ ] El candidato pasó primero por shadow (riesgo cero) antes de recibir tráfico real.
- [ ] Si usa canary, la fracción de tráfico está fijada por adelantado y el reparto real se mide, no se asume.
- [ ] Si usa A/B, el reparto es determinista por `id_cliente` (u otro identificador de cliente estable), no un sorteo por petición.
- [ ] La comparación A/B usa un test estadístico real (valor p, intervalo de confianza), no solo "el número más alto ganó".
- [ ] Hay un criterio de rollback explícito (métrica, versión base, umbral) decidido ANTES de ver el problema.
- [ ] El rollback deja un log real de la decisión (qué, por qué, con qué cifras), sin depender de un reloj de sistema.
- [ ] Ninguna carpeta de trabajo efímera (`proyecto_churn/`) ha quedado dentro del repositorio del curso: verifico con `git status`.

---
## 12. Preguntas de autoevaluación

1. ¿Cuál es la diferencia esencial entre canary y shadow en cuanto a QUIÉN ve la respuesta de la versión candidata?
2. **Cálculo.** Con `UMBRAL_CAIDA_RELATIVA=0.15` y un F1 base de 0,3166, ¿a partir de qué F1 (redondeado a 4 decimales) debería dispararse el rollback? *(Pista: resuelve `(0,3166 - x) / 0,3166 > 0,15` para el valor límite de `x`.)*
3. ¿Por qué el reparto A/B usa un hash del `id_cliente` y no un número aleatorio distinto en cada petición? Explica la consecuencia concreta para la experiencia de un mismo cliente.
4. En la sección 7.2, ¿qué significa que el intervalo de confianza de la diferencia de proporciones NO incluya el 0? ¿Es lo mismo que el valor p sea menor que 0,05?
5. Un compañero propone promover un candidato a producción en cuanto termine un canary de un día sin errores del servidor (sin código 500). ¿Qué le faltaría comprobar, usando el razonamiento de las secciones 6, 7 y 9?

*(Respuesta orientativa a la pregunta 2: `(0,3166 - x) / 0,3166 = 0,15` implica `x = 0,3166 * (1 - 0,15) = 0,26911`; con F1 por debajo de aproximadamente 0,2691 el rollback debería dispararse.)*

---
## Limpieza final

In [ ]:
shutil.rmtree(CARPETA)
print("Carpeta de trabajo eliminada:", CARPETA)
assert not os.path.exists(CARPETA)